# txt2reward — LLM-as-Reward for Highway-RL
PPO روی `highway-v0` با reward signal از Qwen3-4B

## 1. نصب dependencies

In [ ]:
%%capture
!pip install gymnasium highway-env stable-baselines3 transformers accelerate sentencepiece

## 2. Mount Google Drive (مدل Qwen باید اینجا باشه)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
model_path = '/content/drive/MyDrive/models/qwen3-4b'
assert os.path.isdir(model_path), f'Model not found at {model_path}'

## 3. Clone repo

In [ ]:
!git clone https://github.com/A-Nematkhah/txt2reward.git
%cd txt2reward

## 4. تست سریع LLM judge (قبل از training)

In [ ]:
from llm_judge import judge_state

test_cases = [
    (30.0, 2, 25.0, 'expected: good/excellent'),
    (10.0, 0,  3.0, 'expected: crash/dangerous'),
    (40.0, 3,  8.0, 'expected: poor/dangerous'),
]

for speed, lane, dist, label in test_cases:
    r = judge_state(speed, lane, dist)
    print(f'speed={speed}, lane={lane}, dist={dist}m -> reward={r:.3f}  ({label})')

## 5. Training

In [ ]:
!python train.py

## 6. ارزیابی مدل آموزش‌دیده

In [ ]:
import gymnasium as gym
import highway_env
from stable_baselines3 import PPO
from train import make_env

env = make_env()
model = PPO.load('ppo_highway_qwen_reward', env=env)

n_episodes = 5
total_rewards = []

for ep in range(n_episodes):
    obs, _ = env.reset()
    done = False
    ep_reward = 0.0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        ep_reward += reward
        done = terminated or truncated
    total_rewards.append(ep_reward)
    print(f'Episode {ep+1}: total reward = {ep_reward:.2f}')

print(f'\nMean reward over {n_episodes} episodes: {sum(total_rewards)/n_episodes:.2f}')
env.close()